# Mock Stock Contest Simulator

This notebook wraps the simulator engine in `simulator_engine.py`. The core flow is:

1. Parse email-style trade messages.
2. Validate portfolio ID, password, ticker, quantity, and cash/holdings.
3. Execute accepted trades.
4. Apply crisis/news sentiment as market signals.
5. Advance prices using geometric Brownian motion.
6. Recalculate portfolio values and leaderboard ranks.

Secrets such as Gmail passwords and OpenAI keys are intentionally not stored in this notebook.

In [ ]:
from simulator_engine import (
    ContestSimulator,
    EmailMessage,
    EmailOrderParser,
    EmailKind,
    MarketSignal,
    StockConfig,
    UserConfig,
)

In [ ]:
simulator = ContestSimulator(
    stocks=[
        StockConfig('AAA', 'Alpha Corp', initial_price=10.0, shares_outstanding=50_000),
        StockConfig('BBB', 'Beta Corp', initial_price=10.0, shares_outstanding=50_000),
        StockConfig('CCC', 'Gamma Corp', initial_price=10.0, shares_outstanding=50_000),
        StockConfig('DDD', 'Delta Corp', initial_price=10.0, shares_outstanding=50_000),
    ],
    users=[
        UserConfig('team_1', 'pass001', 100_000),
        UserConfig('team_2', 'pass002', 100_000),
        UserConfig('team_3', 'pass003', 100_000),
        UserConfig('team_4', 'pass004', 100_000),
    ],
    seed=7,
)

parser = EmailOrderParser(
    aliases={
        'alpha corp': 'AAA',
        'beta corp': 'BBB',
        'gamma corp': 'CCC',
        'delta corp': 'DDD',
    }
)

In [ ]:
emails = [
    EmailMessage('m1', 'trade', 'Portfolio team_1 password pass001 wants to buy 100 shares of Alhpa Corp.'),
    EmailMessage(
        'm2',
        'trade',
        'portfolio: team_2\npassword: pass002\naction: buy\nstock: BBB\nquantity: 250',
    ),
    EmailMessage(
        'm3',
        'crisis',
        'Rail and tram transport companies face uncertainty, while battlefield medicine demand rises.',
    ),
]

signals_by_message = {
    'm3': [
        MarketSignal('BBB', sentiment=-0.45, volatility_multiplier=1.4, confidence=0.8),
        MarketSignal('CCC', sentiment=0.70, volatility_multiplier=1.2, confidence=0.9),
    ]
}

for email in emails:
    kind = parser.classify(email)
    if kind is EmailKind.MARKET_SIGNAL:
        simulator.apply_market_signals(signals_by_message.get(email.message_id, []))
        print(f'Applied market signal from {email.message_id}')
    elif kind is EmailKind.TRADE_ORDER:
        order = parser.parse_trade(email)
        if order is None:
            print(f'Rejected {email.message_id}: could not parse order')
        else:
            print(simulator.submit_order(order).message)
    elif kind is EmailKind.STOP:
        print('Stop requested')
    else:
        print(f'Ignored {email.message_id}')

In [ ]:
simulator.tick(days=1)
simulator.price_table()

In [ ]:
simulator.leaderboard()